# Lab 06 Web Scrapping

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import ast
from pandas import json_normalize
from tqdm import tqdm
import folium

## Starbucks store locator

- Website: https://www.starbucks.co.uk/store-locator?types=starbucks&latLng=55.8625388%2C-4.284226000000002&zoom=12

- Investigate the website and the feature of the URLs.
- Get the latitudes, longitudes, addresses, Unique IDs, store names, and openning hours of all the Starbucks in the area below as a DataFrame. 
- Visualise the stores on map. 

The boudanries to scrap:

- north_bound = 60.8590 N
- south_bound = 54.6356 N
- west_bound = -7.385 W
- east_bound = 1.7834 E

In [28]:
response = requests.get('https://www.starbucks.co.uk/api/v2/stores/?filter%5Bcoordinates%5D%5Blatitude%5D=55.84370198502933&filter%5Bcoordinates%5D%5Blongitude%5D=-4.246011663724185&filter%5Bradius%5D=100')
response.raise_for_status()

In [31]:
data = response.json()
# df = json_normalize(data)
df_fixed = pd.json_normalize(data['data'], max_level=3)
print(df_fixed.shape)
print(df_fixed.columns.tolist())

(50, 31)
['id', 'type', 'attributes.storeNumber', 'attributes.name', 'attributes.address.streetAddressLine1', 'attributes.address.streetAddressLine2', 'attributes.address.streetAddressLine3', 'attributes.address.city', 'attributes.address.countrySubdivisionCode', 'attributes.address.countryCode', 'attributes.address.postalCode', 'attributes.address.countyCode', 'attributes.phoneNumber', 'attributes.isOpen', 'attributes.open24x7', 'attributes.openHours', 'attributes.features', 'attributes.coordinates.latitude', 'attributes.coordinates.longitude', 'attributes.isOrderingAllowed', 'attributes.todayHours.open', 'attributes.todayHours.open24Hours', 'attributes.todayHours.openAsOfLocalTime', 'attributes.todayHours.openTime', 'attributes.todayHours.closeTime', 'attributes.todayHours.opensIn', 'attributes.todayHours.closesIn', 'attributes.todayHours.localTime', 'attributes.timeZoneInfo.currentTimeOffset', 'attributes.timeZoneInfo.windowsTimeZoneId', 'attributes.timeZoneInfo.olsonTimeZoneId']


In [26]:
print(data.keys())          # see the top level keys
print(type(data['data']))   # likely a list
print(data['data'][0])  

dict_keys(['data'])
<class 'list'>
{'id': '1017974', 'type': 'store', 'attributes': {'storeNumber': '49502-266327', 'name': 'Glasgow - Polmadie Rd DT', 'address': {'streetAddressLine1': '109 Polmadie Rd', 'streetAddressLine2': None, 'streetAddressLine3': None, 'city': 'Glasgow', 'countrySubdivisionCode': 'ENG', 'countryCode': 'GB', 'postalCode': 'G5 0BA', 'countyCode': None}, 'phoneNumber': '01414237945', 'isOpen': True, 'open24x7': None, 'openHours': [{'open': True, 'open24Hours': False, 'openTime': '05:30:00', 'closeTime': '23:00:00', 'date': '2026-02-17T00:00:00.0000000', 'holidayCode': None}, {'open': True, 'open24Hours': False, 'openTime': '05:30:00', 'closeTime': '23:00:00', 'date': '2026-02-18T00:00:00.0000000', 'holidayCode': None}, {'open': True, 'open24Hours': False, 'openTime': '05:30:00', 'closeTime': '23:00:00', 'date': '2026-02-19T00:00:00.0000000', 'holidayCode': None}, {'open': True, 'open24Hours': False, 'openTime': '05:30:00', 'closeTime': '23:00:00', 'date': '2026-02

In [16]:
df = df.join(pd.json_normalize(df.pop('data')))

C:\Users\epicm\AppData\Local\Temp\ipykernel_17984\3340129242.py:1: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  df = df.join(pd.json_normalize(df.pop('data')))


In [21]:
df.head()
df.iloc[0, 0]

{'id': '1017974',
 'type': 'store',
 'attributes.storeNumber': '49502-266327',
 'attributes.name': 'Glasgow - Polmadie Rd DT',
 'attributes.address.streetAddressLine1': '109 Polmadie Rd',
 'attributes.address.streetAddressLine2': None,
 'attributes.address.streetAddressLine3': None,
 'attributes.address.city': 'Glasgow',
 'attributes.address.countrySubdivisionCode': 'ENG',
 'attributes.address.countryCode': 'GB',
 'attributes.address.postalCode': 'G5 0BA',
 'attributes.address.countyCode': None,
 'attributes.phoneNumber': '01414237945',
 'attributes.isOpen': True,
 'attributes.open24x7': None,
 'attributes.openHours': [{'open': True,
   'open24Hours': False,
   'openTime': '05:30:00',
   'closeTime': '23:00:00',
   'date': '2026-02-17T00:00:00.0000000',
   'holidayCode': None},
  {'open': True,
   'open24Hours': False,
   'openTime': '05:30:00',
   'closeTime': '23:00:00',
   'date': '2026-02-18T00:00:00.0000000',
   'holidayCode': None},
  {'open': True,
   'open24Hours': False,
   'o

In [ ]:
df_fixed = pd.DataFrame(df.iloc[0].tolist())
df_fixed.head()

In [25]:
print(len(df.iloc[0].tolist()))
print(df.iloc[0].tolist()[0])
print(len(df.iloc[0].tolist()[0]))
print(df.iloc[0].tolist()[0].keys())

50
{'id': '1017974', 'type': 'store', 'attributes.storeNumber': '49502-266327', 'attributes.name': 'Glasgow - Polmadie Rd DT', 'attributes.address.streetAddressLine1': '109 Polmadie Rd', 'attributes.address.streetAddressLine2': None, 'attributes.address.streetAddressLine3': None, 'attributes.address.city': 'Glasgow', 'attributes.address.countrySubdivisionCode': 'ENG', 'attributes.address.countryCode': 'GB', 'attributes.address.postalCode': 'G5 0BA', 'attributes.address.countyCode': None, 'attributes.phoneNumber': '01414237945', 'attributes.isOpen': True, 'attributes.open24x7': None, 'attributes.openHours': [{'open': True, 'open24Hours': False, 'openTime': '05:30:00', 'closeTime': '23:00:00', 'date': '2026-02-17T00:00:00.0000000', 'holidayCode': None}, {'open': True, 'open24Hours': False, 'openTime': '05:30:00', 'closeTime': '23:00:00', 'date': '2026-02-18T00:00:00.0000000', 'holidayCode': None}, {'open': True, 'open24Hours': False, 'openTime': '05:30:00', 'closeTime': '23:00:00', 'date

In [20]:
df_fixed = pd.json_normalize(df.iloc[0].tolist())
df_fixed.head()

,id,type,attributes.storeNumber,attributes.name,attributes.address.streetAddressLine1,attributes.address.streetAddressLine2,attributes.address.streetAddressLine3,attributes.address.city,attributes.address.countrySubdivisionCode,attributes.address.countryCode,...,attributes.todayHours.open24Hours,attributes.todayHours.openAsOfLocalTime,attributes.todayHours.openTime,attributes.todayHours.closeTime,attributes.todayHours.opensIn,attributes.todayHours.closesIn,attributes.todayHours.localTime,attributes.timeZoneInfo.currentTimeOffset,attributes.timeZoneInfo.windowsTimeZoneId,attributes.timeZoneInfo.olsonTimeZoneId
0,1017974,store,49502-266327,Glasgow - Polmadie Rd DT,109 Polmadie Rd,None,None,Glasgow,ENG,GB,...,False,True,05:30:00,23:00:00,None,5:39:37,2026-02-17T17:20:22.5954099,0,GMT Standard Time,GMT+01:00 Europe/London
1,1017467,store,47633-306329,Glasgow-St Enochs,64 Dunlop Street,None,None,Glasgow,SCT,GB,...,False,True,08:00:00,20:00:00,None,2:39:37,2026-02-17T17:20:22.5954099,0,GMT Standard Time,GMT+01:00 Europe/London
2,1005532,store,18489-191425,Glasgow - Central Station,1 Gordon Street,Glasgow Central Station,None,Glasgow,SCT,GB,...,False,True,06:00:00,20:00:00,None,2:39:37,2026-02-17T17:20:22.5954099,0,GMT Standard Time,GMT+01:00 Europe/London
3,2751,store,12562-60620,Glasgow - Buchanan Street,136 140 Buchanan Street,None,None,Glasgow,SCT,GB,...,False,True,06:30:00,22:00:00,None,4:39:37,2026-02-17T17:20:22.5954099,0,GMT Standard Time,GMT+01:00 Europe/London
4,2823,store,12836-137777,Glasgow - Nelson Mandela Squar,Glasgow West Nile Street,None,58 Nelson Mandela Square,Glasgow,SCT,GB,...,False,True,07:00:00,19:00:00,None,1:39:37,2026-02-17T17:20:22.5954099,0,GMT Standard Time,GMT+01:00 Europe/London
